# Validación Completa de Encuestas Familiares
## Limpieza y Creación de Datos de Prueba

Este notebook realiza una limpieza completa de las encuestas existentes y crea datos de prueba exhaustivos para validar el sistema de gestión familiar de la parroquia.

### Objetivos:
1. 🧹 **Limpieza completa** de todas las encuestas existentes
2. 👨‍👩‍👧‍👦 **Creación de familias completas** con padres, madres e hijos
3. ⚰️ **Registro de difuntos** con relaciones familiares
4. 📊 **Validación de integridad** de datos y relaciones
5. 📈 **Estadísticas y verificación** del sistema completo

### Estructura de Datos:
- **Familias**: Información completa del hogar
- **Personas**: Padres, madres, hijos con datos demográficos
- **Difuntos**: Registros de familiares fallecidos
- **Encuestas**: Datos pastorales y sociales
- **Relaciones**: Vínculos familiares y geográficos

In [ ]:
// 1. Import Required Libraries and Database Connection
import sequelize from './config/sequelize.js';
import { QueryTypes } from 'sequelize';

// Variables globales para el proceso
const API_BASE = 'http://localhost:3000/api';
let authToken = null;

console.log('📚 Librerías importadas correctamente');
console.log('🔗 Configurando conexión a base de datos...');

// Función para autenticación
async function authenticate() {
  try {
    const fetch = (await import('node-fetch')).default;
    
    console.log('🔐 Obteniendo token de autenticación...');
    
    const response = await fetch(`${API_BASE}/auth/login`, {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json'
      },
      body: JSON.stringify({
        correo_electronico: 'admin@parroquia.com',
        contrasena: 'Admin123!'
      })
    });

    const data = await response.json();
    
    if (response.ok && data.data && data.data.accessToken) {
      authToken = data.data.accessToken;
      console.log('✅ Token obtenido exitosamente');
      return true;
    } else {
      console.error('❌ Error al obtener token:', data.message || 'Error desconocido');
      return false;
    }
  } catch (error) {
    console.error('❌ Error en autenticación:', error.message);
    return false;
  }
}

// Verificar conexión a base de datos
async function verificarConexionBD() {
  try {
    console.log('🔍 Verificando conexión a base de datos...');
    await sequelize.authenticate();
    console.log('✅ Conexión a base de datos establecida correctamente');
    return true;
  } catch (error) {
    console.error('❌ Error en conexión a BD:', error.message);
    return false;
  }
}

// Ejecutar verificaciones iniciales
const conexionOK = await verificarConexionBD();
const authOK = await authenticate();

if (!conexionOK || !authOK) {
  throw new Error('No se pudieron establecer las conexiones requeridas');
}

console.log('🎯 Sistema listo para procesar encuestas');
console.log('=====================================');

## 🧹 Limpieza Completa de Datos Existentes

Eliminamos todas las encuestas, familias, personas y difuntos existentes para empezar con una base de datos limpia. Esta operación es irreversible y asegura que no haya conflictos con datos anteriores.

In [ ]:
// 2. Clean Existing Survey Data - Limpieza completa
async function limpiarDatosCompleto() {
  try {
    console.log('🧹 Iniciando limpieza completa de datos...');
    
    // Orden de limpieza respetando restricciones de clave foránea
    const tablasLimpiar = [
      // Tablas relacionales primero
      'persona_destreza',
      'persona_enfermedad', 
      'familia_disposicion_basura',
      'familia_tipo_vivienda',
      'familia_sistema_acueducto',
      'familia_sistema_aguas_residuales',
      'difuntos_familia',
      // Tablas principales después
      'personas',
      'familias',
      'encuestas'
    ];
    
    let totalEliminados = 0;
    
    for (const tabla of tablasLimpiar) {
      try {
        const resultado = await sequelize.query(`DELETE FROM ${tabla}`, {
          type: QueryTypes.DELETE
        });
        
        // Para PostgreSQL, el resultado es un array con el número de filas afectadas
        const filasEliminadas = Array.isArray(resultado) ? resultado[1] : resultado;
        totalEliminados += filasEliminadas || 0;
        
        console.log(`✅ ${tabla}: ${filasEliminadas || 0} registros eliminados`);
      } catch (error) {
        console.warn(`⚠️ Error limpiando ${tabla}:`, error.message);
      }
    }
    
    console.log(`🎯 Total registros eliminados: ${totalEliminados}`);
    
    // Resetear secuencias
    const secuencias = [
      'encuestas_id_encuesta_seq',
      'familias_id_familia_seq',
      'personas_id_personas_seq',
      'difuntos_familia_id_difunto_seq'
    ];
    
    for (const secuencia of secuencias) {
      try {
        await sequelize.query(`SELECT setval('${secuencia}', 1, false)`, {
          type: QueryTypes.SELECT
        });
        console.log(`✅ Secuencia ${secuencia} reseteada`);
      } catch (error) {
        console.warn(`⚠️ Error reseteando secuencia ${secuencia}:`, error.message);
      }
    }
    
    // Verificar limpieza
    const verificacion = await sequelize.query(`
      SELECT 
        'encuestas' as tabla, COUNT(*) as total FROM encuestas
      UNION ALL 
      SELECT 'familias' as tabla, COUNT(*) as total FROM familias
      UNION ALL 
      SELECT 'personas' as tabla, COUNT(*) as total FROM personas
      UNION ALL 
      SELECT 'difuntos_familia' as tabla, COUNT(*) as total FROM difuntos_familia;
    `, { type: QueryTypes.SELECT });
    
    console.log('📊 Verificación de limpieza:');
    verificacion.forEach(row => {
      console.log(`  ${row.tabla}: ${row.total} registros`);
    });
    
    const todosLimpios = verificacion.every(row => parseInt(row.total) === 0);
    
    if (todosLimpios) {
      console.log('✅ ¡Limpieza completa exitosa! Base de datos lista para nuevos datos');
    } else {
      console.warn('⚠️ Algunos registros no fueron eliminados, verificar restricciones');
    }
    
    return todosLimpios;
    
  } catch (error) {
    console.error('❌ Error en limpieza completa:', error);
    throw error;
  }
}

// Ejecutar limpieza
const limpiezaExitosa = await limpiarDatosCompleto();
console.log('=====================================');
console.log(`🎯 Estado de limpieza: ${limpiezaExitosa ? 'EXITOSA' : 'PARCIAL'}`);
console.log('=====================================');

## 👨‍👩‍👧‍👦 Creación de Datos de Referencia y Familias Completas

Creamos datos de referencia necesarios (departamentos, municipios, etc.) y luego generamos familias completas con padres, madres e hijos, incluyendo información demográfica realista.

In [ ]:
// 3. Create Sample People (Padres y Madres) - Crear datos de referencia y familias
async function crearDatosReferencia() {
  try {
    console.log('🏗️ Creando datos de referencia...');
    
    // Crear departamento
    const [departamento] = await sequelize.query(`
      INSERT INTO departamentos (nombre, codigo_dane) 
      VALUES ('Valle del Cauca', '76') 
      ON CONFLICT (codigo_dane) DO UPDATE SET nombre = EXCLUDED.nombre
      RETURNING id_departamento, nombre;
    `, { type: QueryTypes.SELECT });
    
    // Crear municipio
    const [municipio] = await sequelize.query(`
      INSERT INTO municipios (nombre_municipio, codigo_dane, id_departamento) 
      VALUES ('Cali', '76001', ${departamento.id_departamento}) 
      ON CONFLICT (codigo_dane) DO UPDATE SET nombre_municipio = EXCLUDED.nombre_municipio
      RETURNING id_municipio, nombre_municipio;
    `, { type: QueryTypes.SELECT });
    
    // Crear sexos
    await sequelize.query(`
      INSERT INTO sexos (descripcion) VALUES 
      ('Masculino'), ('Femenino')
      ON CONFLICT (descripcion) DO NOTHING;
    `, { type: QueryTypes.INSERT });
    
    const sexos = await sequelize.query(`
      SELECT id_sexo, descripcion FROM sexos ORDER BY id_sexo;
    `, { type: QueryTypes.SELECT });
    
    // Crear tipos de identificación
    await sequelize.query(`
      INSERT INTO tipos_identificacion (nombre, codigo) VALUES 
      ('Cédula de Ciudadanía', 'CC'),
      ('Tarjeta de Identidad', 'TI'),
      ('Registro Civil', 'RC')
      ON CONFLICT (codigo) DO NOTHING;
    `, { type: QueryTypes.INSERT });
    
    const tiposId = await sequelize.query(`
      SELECT id_tipo_identificacion, nombre, codigo FROM tipos_identificacion;
    `, { type: QueryTypes.SELECT });
    
    console.log('✅ Datos de referencia creados:');
    console.log(`  - Departamento: ${departamento.nombre}`);
    console.log(`  - Municipio: ${municipio.nombre_municipio}`);
    console.log(`  - Sexos: ${sexos.length}`);
    console.log(`  - Tipos ID: ${tiposId.length}`);
    
    return { departamento, municipio, sexos, tiposId };
    
  } catch (error) {
    console.error('❌ Error creando datos de referencia:', error);
    throw error;
  }
}

// Crear familias completas con padres, madres, hijos
async function crearFamiliasCompletas(datosRef) {
  try {
    console.log('👨‍👩‍👧‍👦 Creando familias completas...');
    
    const familias = [];
    const personas = [];
    const timestamp = new Date().toISOString().split('T')[0];
    
    // Definir 5 familias diferentes
    const datosFamilias = [
      {
        apellido: 'García Rodríguez',
        direccion: 'Calle 15 #23-45, Barrio El Poblado',
        telefono: '3201234567',
        email: 'garcia.familia@email.com',
        miembros: [
          { nombre: 'Carlos Alberto', apellidos: 'García Pérez', rol: 'padre', edad: 45, identificacion: '16789234' },
          { nombre: 'María Elena', apellidos: 'Rodríguez López', rol: 'madre', edad: 42, identificacion: '41567890' },
          { nombre: 'Juan Carlos', apellidos: 'García Rodríguez', rol: 'hijo', edad: 18, identificacion: '1007654321' },
          { nombre: 'Ana Sofía', apellidos: 'García Rodríguez', rol: 'hija', edad: 15, identificacion: '1234567890' }
        ]
      },
      {
        apellido: 'Martínez Silva',
        direccion: 'Carrera 8 #12-34, Barrio San Fernando',
        telefono: '3109876543',
        email: 'martinez.familia@email.com',
        miembros: [
          { nombre: 'José Manuel', apellidos: 'Martínez González', rol: 'padre', edad: 38, identificacion: '17890345' },
          { nombre: 'Laura Patricia', apellidos: 'Silva Ramírez', rol: 'madre', edad: 36, identificacion: '42678901' },
          { nombre: 'Camila', apellidos: 'Martínez Silva', rol: 'hija', edad: 12, identificacion: '1098765432' },
          { nombre: 'Santiago', apellidos: 'Martínez Silva', rol: 'hijo', edad: 8, identificacion: '1345678901' }
        ]
      },
      {
        apellido: 'López Vargas',
        direccion: 'Avenida 6 #45-67, Barrio La Flora',
        telefono: '3157654321',
        email: 'lopez.familia@email.com',
        miembros: [
          { nombre: 'Roberto', apellidos: 'López Herrera', rol: 'padre', edad: 41, identificacion: '18901456' },
          { nombre: 'Carmen Rosa', apellidos: 'Vargas Mejía', rol: 'madre', edad: 39, identificacion: '43789012' },
          { nombre: 'Alejandra', apellidos: 'López Vargas', rol: 'hija', edad: 16, identificacion: '1456789012' },
          { nombre: 'Diego Andrés', apellidos: 'López Vargas', rol: 'hijo', edad: 13, identificacion: '1567890123' }
        ]
      },
      {
        apellido: 'Hernández Torres',
        direccion: 'Calle 25 #34-56, Barrio Normandía',
        telefono: '3048765432',
        email: 'hernandez.familia@email.com',
        miembros: [
          { nombre: 'Miguel Ángel', apellidos: 'Hernández Castro', rol: 'padre', edad: 43, identificacion: '19012567' },
          { nombre: 'Claudia Marcela', apellidos: 'Torres Jiménez', rol: 'madre', edad: 40, identificacion: '44890123' },
          { nombre: 'Valentina', apellidos: 'Hernández Torres', rol: 'hija', edad: 14, identificacion: '1678901234' },
          { nombre: 'Sebastián', apellidos: 'Hernández Torres', rol: 'hijo', edad: 11, identificacion: '1789012345' }
        ]
      },
      {
        apellido: 'Gómez Ruiz',
        direccion: 'Carrera 12 #56-78, Barrio El Ingenio',
        telefono: '3186543210',
        email: 'gomez.familia@email.com',
        miembros: [
          { nombre: 'Fernando', apellidos: 'Gómez Morales', rol: 'padre', edad: 47, identificacion: '20123678' },
          { nombre: 'Patricia Isabel', apellidos: 'Ruiz Sandoval', rol: 'madre', edad: 44, identificacion: '45901234' },
          { nombre: 'Isabella', apellidos: 'Gómez Ruiz', rol: 'hija', edad: 17, identificacion: '1890123456' },
          { nombre: 'Mateo', apellidos: 'Gómez Ruiz', rol: 'hijo', edad: 9, identificacion: '1901234567' }
        ]
      }
    ];
    
    // Crear cada familia
    for (let i = 0; i < datosFamilias.length; i++) {
      const fam = datosFamilias[i];
      
      // Crear familia
      const [familiaCreada] = await sequelize.query(`
        INSERT INTO familias (
          apellido_familiar, direccion_familia, numero_contacto, telefono, email,
          tamaño_familia, tipo_vivienda, estado_encuesta, fecha_ultima_encuesta,
          id_municipio, comunionEnCasa, observaciones
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        RETURNING *;
      `, {
        replacements: [
          fam.apellido,
          fam.direccion,
          fam.telefono,
          fam.telefono,
          fam.email,
          fam.miembros.length,
          'Casa propia',
          'active',
          timestamp,
          datosRef.municipio.id_municipio,
          true,
          `Familia ${fam.apellido} - Datos de prueba completos`
        ],
        type: QueryTypes.SELECT
      });
      
      familias.push(familiaCreada);
      console.log(`✅ Familia ${i+1}: ${fam.apellido} (ID: ${familiaCreada.id_familia})`);
      
      // Crear miembros de la familia
      for (const miembro of fam.miembros) {
        const sexoId = miembro.rol === 'padre' || miembro.rol === 'hijo' ? 
          datosRef.sexos.find(s => s.descripcion === 'Masculino').id_sexo :
          datosRef.sexos.find(s => s.descripcion === 'Femenino').id_sexo;
        
        const tipoIdObj = datosRef.tiposId.find(t => t.codigo === 'CC') || datosRef.tiposId[0];
        
        const fechaNacimiento = new Date();
        fechaNacimiento.setFullYear(fechaNacimiento.getFullYear() - miembro.edad);
        
        const [personaCreada] = await sequelize.query(`
          INSERT INTO personas (
            primer_nombre, primer_apellido, segundo_apellido, identificacion,
            fecha_nacimiento, telefono, correo_electronico, direccion,
            id_sexo, id_tipo_identificacion_tipo_identificacion, id_familia_familias
          ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
          RETURNING *;
        `, {
          replacements: [
            miembro.nombre,
            miembro.apellidos.split(' ')[0],
            miembro.apellidos.split(' ')[1],
            miembro.identificacion,
            fechaNacimiento.toISOString().split('T')[0],
            fam.telefono,
            `${miembro.nombre.toLowerCase().replace(' ', '.')}.${miembro.apellidos.split(' ')[0].toLowerCase()}@email.com`,
            fam.direccion,
            sexoId,
            tipoIdObj.id_tipo_identificacion,
            familiaCreada.id_familia
          ],
          type: QueryTypes.SELECT
        });
        
        personas.push({...personaCreada, rol: miembro.rol, familia_apellido: fam.apellido});
      }
    }
    
    console.log(`✅ ${familias.length} familias creadas con ${personas.length} personas`);
    
    // Mostrar estadísticas
    const padres = personas.filter(p => p.rol === 'padre');
    const madres = personas.filter(p => p.rol === 'madre');
    const hijos = personas.filter(p => p.rol === 'hijo' || p.rol === 'hija');
    
    console.log('📊 Estadísticas de personas creadas:');
    console.log(`  - Padres: ${padres.length}`);
    console.log(`  - Madres: ${madres.length}`);
    console.log(`  - Hijos: ${hijos.length}`);
    console.log(`  - Total: ${personas.length}`);
    
    return { familias, personas, padres, madres, hijos };
    
  } catch (error) {
    console.error('❌ Error creando familias completas:', error);
    throw error;
  }
}

// Ejecutar creación de datos
const datosReferencia = await crearDatosReferencia();
const familiasCreadas = await crearFamiliasCompletas(datosReferencia);

console.log('=====================================');
console.log('🎯 Familias y personas creadas exitosamente');
console.log('=====================================');

## ⚰️ Creación de Registros de Difuntos

Creamos registros de familiares fallecidos para cada familia, estableciendo relaciones familiares realistas y datos de fallecimiento con fechas y causas apropiadas.

In [ ]:
// 4. Create Sample Deceased Records (Difuntos)
async function crearRegistrosDifuntos(familias) {
  try {
    console.log('⚰️ Creando registros de difuntos...');
    
    const difuntos = [];
    
    // Datos realistas de familiares fallecidos
    const datosDifuntos = [
      {
        familia_idx: 0,
        nombre: 'Roberto García Mendoza',
        parentesco: 'Abuelo paterno',
        fecha_fallecimiento: '2018-03-15',
        causa: 'Enfermedad cardiovascular',
        observaciones: 'Patriarca de la familia García, muy querido por todos'
      },
      {
        familia_idx: 0,
        nombre: 'Elena Pérez Vásquez',
        parentesco: 'Abuela paterna',
        fecha_fallecimiento: '2020-11-08',
        causa: 'Complicaciones respiratorias',
        observaciones: 'Dedicó su vida a la familia y la comunidad'
      },
      {
        familia_idx: 1,
        nombre: 'Manuel Martínez Torres',
        parentesco: 'Abuelo paterno',
        fecha_fallecimiento: '2019-07-22',
        causa: 'Cáncer',
        observaciones: 'Luchó valientemente contra la enfermedad'
      },
      {
        familia_idx: 1,
        nombre: 'Rosa Silva Jiménez',
        parentesco: 'Abuela materna',
        fecha_fallecimiento: '2021-01-30',
        causa: 'Edad avanzada',
        observaciones: 'Falleció en paz rodeada de su familia'
      },
      {
        familia_idx: 2,
        nombre: 'Carlos López Herrera',
        parentesco: 'Tío paterno',
        fecha_fallecimiento: '2022-05-12',
        causa: 'Accidente',
        observaciones: 'Partida inesperada que conmocionó a la familia'
      },
      {
        familia_idx: 2,
        nombre: 'Mercedes Vargas Luna',
        parentesco: 'Abuela materna',
        fecha_fallecimiento: '2017-12-03',
        causa: 'Diabetes complicada',
        observaciones: 'Gran ejemplo de fortaleza y fe'
      },
      {
        familia_idx: 3,
        nombre: 'Antonio Hernández Ruiz',
        parentesco: 'Abuelo paterno',
        fecha_fallecimiento: '2020-09-18',
        causa: 'COVID-19',
        observaciones: 'Víctima de la pandemia, muy extrañado por la familia'
      },
      {
        familia_idx: 3,
        nombre: 'María Torres Gómez',
        parentesco: 'Bisabuela materna',
        fecha_fallecimiento: '2016-04-25',
        causa: 'Edad avanzada (95 años)',
        observaciones: 'Vivió una vida plena y dejó grandes enseñanzas'
      },
      {
        familia_idx: 4,
        nombre: 'Luis Gómez Ortiz',
        parentesco: 'Hermano del padre',
        fecha_fallecimiento: '2021-08-14',
        causa: 'Infarto al miocardio',
        observaciones: 'Partida repentina, muy querido por los sobrinos'
      },
      {
        familia_idx: 4,
        nombre: 'Carmen Ruiz Moreno',
        parentesco: 'Abuela materna',
        fecha_fallecimiento: '2019-02-28',
        causa: 'Alzheimer avanzado',
        observaciones: 'Tras una larga batalla contra la enfermedad'
      }
    ];
    
    // Crear registros de difuntos
    for (const difunto of datosDifuntos) {
      const familia = familias[difunto.familia_idx];
      
      const [difuntoCreado] = await sequelize.query(`
        INSERT INTO difuntos_familia (
          nombre_completo, fecha_fallecimiento, observaciones, id_familia_familias
        ) VALUES (?, ?, ?, ?)
        RETURNING *;
      `, {
        replacements: [
          `${difunto.nombre} (${difunto.parentesco})`,
          difunto.fecha_fallecimiento,
          `${difunto.causa}. ${difunto.observaciones}`,
          familia.id_familia
        ],
        type: QueryTypes.SELECT
      });
      
      difuntos.push({
        ...difuntoCreado,
        familia_apellido: familia.apellido_familiar,
        parentesco: difunto.parentesco,
        causa: difunto.causa
      });
    }
    
    console.log(`✅ ${difuntos.length} registros de difuntos creados`);
    
    // Mostrar estadísticas por familia
    console.log('📊 Difuntos por familia:');
    familias.forEach((familia, idx) => {
      const difuntosFamilia = difuntos.filter(d => d.id_familia_familias === familia.id_familia);
      console.log(`  - ${familia.apellido_familiar}: ${difuntosFamilia.length} difuntos`);
    });
    
    // Estadísticas por tipo de parentesco
    const parentescos = {};
    difuntos.forEach(d => {
      if (parentescos[d.parentesco]) {
        parentescos[d.parentesco]++;
      } else {
        parentescos[d.parentesco] = 1;
      }
    });
    
    console.log('📈 Distribución por parentesco:');
    Object.entries(parentescos).forEach(([parentesco, cantidad]) => {
      console.log(`  - ${parentesco}: ${cantidad}`);
    });
    
    // Estadísticas por década de fallecimiento
    const decadas = {};
    difuntos.forEach(d => {
      const año = new Date(d.fecha_fallecimiento).getFullYear();
      const decada = `${Math.floor(año / 10) * 10}s`;
      if (decadas[decada]) {
        decadas[decada]++;
      } else {
        decadas[decada] = 1;
      }
    });
    
    console.log('📅 Distribución por década:');
    Object.entries(decadas).forEach(([decada, cantidad]) => {
      console.log(`  - ${decada}: ${cantidad} fallecimientos`);
    });
    
    return difuntos;
    
  } catch (error) {
    console.error('❌ Error creando registros de difuntos:', error);
    throw error;
  }
}

// Ejecutar creación de difuntos
const difuntosCreados = await crearRegistrosDifuntos(familiasCreadas.familias);

console.log('=====================================');
console.log('🎯 Registros de difuntos creados exitosamente');
console.log(`⚰️ Total difuntos: ${difuntosCreados.length}`);
console.log('=====================================');

## 📝 Creación de Encuestas (Surveys)

Ahora crearemos las encuestas reales que vinculan las familias con la información pastoral y social. Cada encuesta incluirá:

- **Información pastoral**: Sacramentos, participación religiosa, necesidades espirituales
- **Información social**: Condiciones económicas, salud, educación, vivienda
- **Relación con difuntos**: Conexión con los registros de familiares fallecidos
- **Datos geográficos**: Ubicación específica y contacto

In [ ]:
// 5. Create Complete Survey Records (Encuestas)
async function crearEncuestasCompletas(familias, difuntos) {
  try {
    console.log('📝 Creando encuestas completas...');
    
    const encuestas = [];
    
    // Datos realistas de encuestas por familia
    const datosEncuestas = [
      {
        familia_idx: 0,
        info_pastoral: {
          bautizados: 4,
          primera_comunion: 3,
          confirmados: 2,
          matrimonio_religioso: true,
          participacion_misa: 'Semanal',
          ministerios: 'Coro parroquial, Catequesis',
          necesidades_espirituales: 'Formación en valores cristianos para los jóvenes'
        },
        info_social: {
          ingresos_mensuales: 2800000,
          tipo_vivienda: 'Propia',
          servicios_publicos: 'Completos',
          educacion_promedio: 'Secundaria completa',
          problemas_salud: 'Diabetes (padre), ninguna grave',
          empleos: 'Comerciante, Ama de casa, Estudiantes'
        },
        contacto: {
          telefono: '3001234567',
          direccion: 'Calle 15 #23-45, Barrio Centro',
          referencias: 'Casa esquina, pintura amarilla'
        }
      },
      {
        familia_idx: 1,
        info_pastoral: {
          bautizados: 3,
          primera_comunion: 2,
          confirmados: 2,
          matrimonio_religioso: true,
          participacion_misa: 'Ocasional',
          ministerios: 'Ninguno actualmente',
          necesidades_espirituales: 'Retomar la vida sacramental, preparación matrimonial para hijos'
        },
        info_social: {
          ingresos_mensuales: 3200000,
          tipo_vivienda: 'Propia',
          servicios_publicos: 'Completos',
          educacion_promedio: 'Técnica',
          problemas_salud: 'Hipertensión (madre)',
          empleos: 'Técnico electricista, Enfermera, Estudiante'
        },
        contacto: {
          telefono: '3109876543',
          direccion: 'Carrera 8 #12-30, Sector La Esperanza',
          referencias: 'Frente al colegio, portón verde'
        }
      },
      {
        familia_idx: 2,
        info_pastoral: {
          bautizados: 5,
          primera_comunion: 4,
          confirmados: 3,
          matrimonio_religioso: true,
          participacion_misa: 'Diaria',
          ministerios: 'Eucaristía, Pastoral social, Grupo de oración',
          necesidades_espirituales: 'Continuar fortalecimiento espiritual, evangelización familiar'
        },
        info_social: {
          ingresos_mensuales: 4500000,
          tipo_vivienda: 'Propia',
          servicios_publicos: 'Completos',
          educacion_promedio: 'Universitaria',
          problemas_salud: 'Ninguna grave',
          empleos: 'Profesor, Contadora, Estudiantes universitarios'
        },
        contacto: {
          telefono: '3158765432',
          direccion: 'Avenida Principal #45-67, Urbanización San José',
          referencias: 'Casa de dos pisos, jardín frontal'
        }
      },
      {
        familia_idx: 3,
        info_pastoral: {
          bautizados: 3,
          primera_comunion: 1,
          confirmados: 1,
          matrimonio_religioso: false,
          participacion_misa: 'Mensual',
          ministerios: 'Ninguno',
          necesidades_espirituales: 'Preparación sacramental para los hijos, regularización matrimonial'
        },
        info_social: {
          ingresos_mensuales: 1800000,
          tipo_vivienda: 'Arrendada',
          servicios_publicos: 'Básicos (agua, luz)',
          educacion_promedio: 'Primaria completa',
          problemas_salud: 'Asma (hijo menor)',
          empleos: 'Construcción, Oficios varios'
        },
        contacto: {
          telefono: '3187654321',
          direccion: 'Calle 5 #8-12, Sector Popular',
          referencias: 'Apartamento segundo piso, escalera externa'
        }
      },
      {
        familia_idx: 4,
        info_pastoral: {
          bautizados: 4,
          primera_comunion: 3,
          confirmados: 2,
          matrimonio_religioso: true,
          participacion_misa: 'Quincenal',
          ministerios: 'Pastoral juvenil',
          necesidades_espirituales: 'Acompañamiento espiritual, fortalecimiento de la fe en crisis'
        },
        info_social: {
          ingresos_mensuales: 2500000,
          tipo_vivienda: 'Propia (heredada)',
          servicios_publicos: 'Completos',
          educacion_promedio: 'Secundaria',
          problemas_salud: 'Depresión (madre, por duelos familiares)',
          empleos: 'Mecánico, Estilista, Estudiantes'
        },
        contacto: {
          telefono: '3176543210',
          direccion: 'Calle 20 #15-25, Barrio Tradicional',
          referencias: 'Casa antigua, patio grande, cerca de la iglesia'
        }
      }
    ];
    
    // Crear registros de encuesta
    for (let i = 0; i < datosEncuestas.length; i++) {
      const familia = familias[i];
      const datosEncuesta = datosEncuestas[i];
      const difuntosFamilia = difuntos.filter(d => d.id_familia_familias === familia.id_familia);
      
      // Preparar información completa de la encuesta
      const infoCompleta = {
        pastoral: datosEncuesta.info_pastoral,
        social: datosEncuesta.info_social,
        difuntos: difuntosFamilia.map(d => ({
          nombre: d.nombre_completo,
          fecha_fallecimiento: d.fecha_fallecimiento,
          observaciones: d.observaciones
        }))
      };
      
      const fechaEncuesta = new Date();
      fechaEncuesta.setDate(fechaEncuesta.getDate() - Math.floor(Math.random() * 30)); // Últimos 30 días
      
      const [encuestaCreada] = await sequelize.query(`
        INSERT INTO encuestas (
          fecha_encuesta,
          observaciones_generales,
          telefono,
          direccion,
          referencias,
          informacion_adicional,
          id_familia_familias,
          created_at,
          updated_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, NOW(), NOW())
        RETURNING *;
      `, {
        replacements: [
          fechaEncuesta.toISOString().split('T')[0],
          `Familia ${familia.apellido_familiar}: ${datosEncuesta.info_pastoral.necesidades_espirituales}`,
          datosEncuesta.contacto.telefono,
          datosEncuesta.contacto.direccion,
          datosEncuesta.contacto.referencias,
          JSON.stringify(infoCompleta),
          familia.id_familia
        ],
        type: QueryTypes.SELECT
      });
      
      encuestas.push({
        ...encuestaCreada,
        familia_apellido: familia.apellido_familiar,
        miembros_familia: familia.miembros_count,
        difuntos_count: difuntosFamilia.length,
        info_pastoral: datosEncuesta.info_pastoral,
        info_social: datosEncuesta.info_social
      });
    }
    
    console.log(`✅ ${encuestas.length} encuestas completas creadas`);
    
    // Estadísticas detalladas
    console.log('📊 Estadísticas de encuestas:');
    
    // Estadísticas sacramentales
    const totalBautizados = encuestas.reduce((sum, e) => sum + e.info_pastoral.bautizados, 0);
    const totalPrimeraComunion = encuestas.reduce((sum, e) => sum + e.info_pastoral.primera_comunion, 0);
    const totalConfirmados = encuestas.reduce((sum, e) => sum + e.info_pastoral.confirmados, 0);
    const matrimoniosReligiosos = encuestas.filter(e => e.info_pastoral.matrimonio_religioso).length;
    
    console.log('  Sacramentos:');
    console.log(`    - Total bautizados: ${totalBautizados}`);
    console.log(`    - Primera comunión: ${totalPrimeraComunion}`);
    console.log(`    - Confirmados: ${totalConfirmados}`);
    console.log(`    - Matrimonios religiosos: ${matrimoniosReligiosos}/${encuestas.length}`);
    
    // Estadísticas sociales
    const ingresoPromedio = encuestas.reduce((sum, e) => sum + e.info_social.ingresos_mensuales, 0) / encuestas.length;
    const viviendaPropia = encuestas.filter(e => e.info_social.tipo_vivienda === 'Propia').length;
    
    console.log('  Condiciones sociales:');
    console.log(`    - Ingreso promedio: $${ingresoPromedio.toLocaleString()}`);
    console.log(`    - Vivienda propia: ${viviendaPropia}/${encuestas.length}`);
    
    // Participación religiosa
    const participacion = {};
    encuestas.forEach(e => {
      const freq = e.info_pastoral.participacion_misa;
      participacion[freq] = (participacion[freq] || 0) + 1;
    });
    
    console.log('  Participación en misa:');
    Object.entries(participacion).forEach(([freq, count]) => {
      console.log(`    - ${freq}: ${count} familias`);
    });
    
    // Resumen por familia
    console.log('👨‍👩‍👧‍👦 Resumen por familia:');
    encuestas.forEach((encuesta, idx) => {
      console.log(`  ${idx + 1}. ${encuesta.familia_apellido}:`);
      console.log(`     - Miembros: ${encuesta.miembros_familia}, Difuntos: ${encuesta.difuntos_count}`);
      console.log(`     - Ingresos: $${encuesta.info_social.ingresos_mensuales.toLocaleString()}`);
      console.log(`     - Participación: ${encuesta.info_pastoral.participacion_misa}`);
      console.log(`     - Ministerios: ${encuesta.info_pastoral.ministerios || 'Ninguno'}`);
    });
    
    return encuestas;
    
  } catch (error) {
    console.error('❌ Error creando encuestas:', error);
    throw error;
  }
}

// Ejecutar creación de encuestas
const encuestasCreadas = await crearEncuestasCompletas(familiasCreadas.familias, difuntosCreados);

console.log('=====================================');
console.log('🎯 Sistema de encuestas configurado completamente');
console.log(`📝 Total encuestas: ${encuestasCreadas.length}`);
console.log(`👨‍👩‍👧‍👦 Total familias: ${familiasCreadas.familias.length}`);
console.log(`👤 Total personas: ${familiasCreadas.totalPersonas}`);
console.log(`⚰️ Total difuntos: ${difuntosCreados.length}`);
console.log('=====================================');

## ✅ Validación y Verificación Final

Realizamos una validación completa del sistema de encuestas para asegurar que:

1. **Integridad referencial**: Todas las relaciones entre tablas están correctas
2. **Completitud de datos**: Cada familia tiene personas, algunas tienen difuntos, todas tienen encuestas
3. **Consistencia**: Los datos son coherentes y realistas
4. **Funcionalidad**: El sistema está listo para pruebas de servicios y APIs

In [ ]:
// 6. Final Validation and System Verification
async function validacionCompleta() {
  try {
    console.log('🔍 Realizando validación completa del sistema...');
    
    // 1. Verificar consistencia de datos
    console.log('\n📊 1. VERIFICACIÓN DE CONSISTENCIA:');
    
    const [estadisticas] = await sequelize.query(`
      SELECT 
        (SELECT COUNT(*) FROM familias) as total_familias,
        (SELECT COUNT(*) FROM personas) as total_personas,
        (SELECT COUNT(*) FROM difuntos_familia) as total_difuntos,  
        (SELECT COUNT(*) FROM encuestas) as total_encuestas
    `, { type: QueryTypes.SELECT });
    
    console.log(`✅ Familias: ${estadisticas.total_familias}`);
    console.log(`✅ Personas: ${estadisticas.total_personas}`);
    console.log(`✅ Difuntos: ${estadisticas.total_difuntos}`);
    console.log(`✅ Encuestas: ${estadisticas.total_encuestas}`);
    
    // 2. Verificar integridad referencial
    console.log('\n🔗 2. VERIFICACIÓN DE INTEGRIDAD REFERENCIAL:');
    
    const [integridadPersonas] = await sequelize.query(`
      SELECT COUNT(*) as huerfanas 
      FROM personas p 
      LEFT JOIN familias f ON p.id_familia_familias = f.id_familia 
      WHERE f.id_familia IS NULL
    `, { type: QueryTypes.SELECT });
    
    const [integridadDifuntos] = await sequelize.query(`
      SELECT COUNT(*) as huerfanos 
      FROM difuntos_familia d 
      LEFT JOIN familias f ON d.id_familia_familias = f.id_familia 
      WHERE f.id_familia IS NULL
    `, { type: QueryTypes.SELECT });
    
    const [integridadEncuestas] = await sequelize.query(`
      SELECT COUNT(*) as huerfanas 
      FROM encuestas e 
      LEFT JOIN familias f ON e.id_familia_familias = f.id_familia 
      WHERE f.id_familia IS NULL
    `, { type: QueryTypes.SELECT });
    
    console.log(`✅ Personas sin familia: ${integridadPersonas.huerfanas} (debe ser 0)`);
    console.log(`✅ Difuntos sin familia: ${integridadDifuntos.huerfanos} (debe ser 0)`);
    console.log(`✅ Encuestas sin familia: ${integridadEncuestas.huerfanas} (debe ser 0)`);
    
    // 3. Análisis demográfico detallado
    console.log('\n👥 3. ANÁLISIS DEMOGRÁFICO:');
    
    const demografico = await sequelize.query(`
      SELECT 
        f.apellido_familiar,
        COUNT(p.id_persona) as miembros,
        COUNT(d.id_difunto) as difuntos,
        STRING_AGG(DISTINCT 
          CASE 
            WHEN p.relacion_jefe_hogar IN ('Padre', 'Jefe de hogar') THEN 'Padre'
            WHEN p.relacion_jefe_hogar IN ('Madre', 'Esposa') THEN 'Madre'
            ELSE NULL
          END, ', ') as padres_presentes
      FROM familias f
      LEFT JOIN personas p ON f.id_familia = p.id_familia_familias
      LEFT JOIN difuntos_familia d ON f.id_familia = d.id_familia_familias
      GROUP BY f.id_familia, f.apellido_familiar
      ORDER BY f.apellido_familiar
    `, { type: QueryTypes.SELECT });
    
    demografico.forEach((familia, idx) => {
      console.log(`  ${idx + 1}. ${familia.apellido_familiar}:`);
      console.log(`     - Miembros vivos: ${familia.miembros}`);
      console.log(`     - Difuntos: ${familia.difuntos}`);
      console.log(`     - Padres: ${familia.padres_presentes || 'No identificados'}`);
    });
    
    // 4. Verificar estructura de edades realista
    console.log('\n📈 4. DISTRIBUCIÓN DE EDADES:');
    
    const edades = await sequelize.query(`
      SELECT 
        CASE 
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 18 THEN 'Menores (0-17)'
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 35 THEN 'Jóvenes (18-34)'
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 60 THEN 'Adultos (35-59)'
          ELSE 'Mayores (60+)'
        END as grupo_edad,
        COUNT(*) as cantidad
      FROM personas
      GROUP BY 
        CASE 
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 18 THEN 'Menores (0-17)'
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 35 THEN 'Jóvenes (18-34)'
          WHEN EXTRACT(YEAR FROM AGE(fecha_nacimiento)) < 60 THEN 'Adultos (35-59)'
          ELSE 'Mayores (60+)'
        END
      ORDER BY MIN(EXTRACT(YEAR FROM AGE(fecha_nacimiento)))
    `, { type: QueryTypes.SELECT });
    
    edades.forEach(grupo => {
      console.log(`  ✅ ${grupo.grupo_edad}: ${grupo.cantidad} personas`);
    });
    
    // 5. Verificar información pastoral y social
    console.log('\n⛪ 5. INFORMACIÓN PASTORAL Y SOCIAL:');
    
    const encuestasInfo = await sequelize.query(`
      SELECT 
        e.id_encuesta,
        f.apellido_familiar,
        e.informacion_adicional::text as info_json,
        e.telefono,
        e.direccion
      FROM encuestas e
      JOIN familias f ON e.id_familia_familias = f.id_familia
      ORDER BY f.apellido_familiar
    `, { type: QueryTypes.SELECT });
    
    let totalBautizados = 0;
    let familiasConMinisterios = 0;
    let ingresoTotal = 0;
    
    encuestasInfo.forEach((encuesta, idx) => {
      try {
        const info = JSON.parse(encuesta.info_json);
        totalBautizados += info.pastoral?.bautizados || 0;
        if (info.pastoral?.ministerios && info.pastoral.ministerios !== 'Ninguno') {
          familiasConMinisterios++;
        }
        ingresoTotal += info.social?.ingresos_mensuales || 0;
        
        console.log(`  ${idx + 1}. ${encuesta.apellido_familiar}:`);
        console.log(`     - Teléfono: ${encuesta.telefono}`);
        console.log(`     - Bautizados: ${info.pastoral?.bautizados || 0}`);
        console.log(`     - Ingresos: $${(info.social?.ingresos_mensuales || 0).toLocaleString()}`);
      } catch (error) {
        console.log(`  ${idx + 1}. ${encuesta.apellido_familiar}: Error parsing JSON`);
      }
    });
    
    console.log(`\n📊 RESUMEN PASTORAL Y SOCIAL:`);
    console.log(`  ✅ Total bautizados reportados: ${totalBautizados}`);
    console.log(`  ✅ Familias con ministerios: ${familiasConMinisterios}/${encuestasInfo.length}`);
    console.log(`  ✅ Ingreso promedio familiar: $${Math.round(ingresoTotal / encuestasInfo.length).toLocaleString()}`);
    
    // 6. Verificación final de sistema
    console.log('\n🎯 6. VERIFICACIÓN FINAL DEL SISTEMA:');
    
    const todoCorrecto = 
      estadisticas.total_familias === 5 &&
      estadisticas.total_personas === 20 &&
      estadisticas.total_difuntos === 10 &&
      estadisticas.total_encuestas === 5 &&
      integridadPersonas.huerfanas === 0 &&
      integridadDifuntos.huerfanos === 0 &&
      integridadEncuestas.huerfanas === 0;
    
    if (todoCorrecto) {
      console.log('🟢 ¡SISTEMA COMPLETAMENTE FUNCIONAL!');
      console.log('   - Todas las tablas tienen datos consistentes');
      console.log('   - Integridad referencial perfecta');
      console.log('   - Familias con padres, madres, hijos e información completa');
      console.log('   - Registros de difuntos vinculados correctamente');
      console.log('   - Encuestas con información pastoral y social detallada');
      console.log('   - Sistema listo para pruebas de APIs y servicios');
    } else {
      console.log('🟡 SISTEMA CON ADVERTENCIAS:');
      console.log('   - Revisar consistencia de datos');
      console.log('   - Verificar integridad referencial');
    }
    
    // 7. Información para pruebas de API
    console.log('\n🔧 7. INFORMACIÓN PARA PRUEBAS DE API:');
    console.log('   Endpoints sugeridos para probar:');
    console.log('   - GET /api/familias - Listar todas las familias');
    console.log('   - GET /api/personas - Listar todas las personas');
    console.log('   - GET /api/encuestas - Listar todas las encuestas');
    console.log('   - GET /api/difuntos - Listar registros de difuntos');
    console.log('   - GET /api/consolidados/familias - Servicio de familias consolidadas');
    
    const familiaEjemplo = demografico[0];
    console.log(`\n   Familia de ejemplo para pruebas: "${familiaEjemplo.apellido_familiar}"`);
    console.log(`   - ${familiaEjemplo.miembros} miembros vivos`);
    console.log(`   - ${familiaEjemplo.difuntos} familiares difuntos`);
    
    return {
      exito: todoCorrecto,
      estadisticas,
      mensaje: todoCorrecto ? 'Sistema completamente funcional' : 'Sistema con advertencias'
    };
    
  } catch (error) {
    console.error('❌ ERROR en validación completa:', error);
    return {
      exito: false,
      error: error.message,
      mensaje: 'Error durante la validación'
    };
  }
}

// Ejecutar validación completa
const resultadoValidacion = await validacionCompleta();

console.log('\n=====================================');
console.log('🏁 VALIDACIÓN COMPLETADA');
console.log(`📊 Estado: ${resultadoValidacion.exito ? '✅ EXITOSA' : '❌ CON ERRORES'}`);
console.log(`💬 Resultado: ${resultadoValidacion.mensaje}`);
console.log('=====================================');

// Mostrar resumen final
if (resultadoValidacion.exito) {
  console.log('\n🎉 ¡SISTEMA DE ENCUESTAS LISTO!');
  console.log('   El sistema tiene:');
  console.log('   ✅ 5 familias completas con apellidos realistas');
  console.log('   ✅ 20 personas con padres, madres e hijos');
  console.log('   ✅ 10 registros de difuntos con relaciones familiares');
  console.log('   ✅ 5 encuestas con información pastoral y social completa');
  console.log('   ✅ Integridad referencial perfecta');
  console.log('   ✅ Datos demográficos realistas');
  console.log('\n   🚀 ¡Listo para probar servicios y APIs!');
}